# 05 - Full Release Pipeline\n\nMaster pipeline: Validate -> Smoke -> Test -> Export All -> Package -> Download.\n\n**Runtime: ~45 min**\n\nStops on first failure so you don't waste compute on broken exports.

In [ ]:
VERSION = '0.4.0'  # Change this each release

In [ ]:
!bash scripts/colab/setup_colab.sh

In [ ]:
import sys, os, time, json\nsys.path.insert(0, 'scripts/colab')\nfrom godot_runner import GodotRunner\n\nrunner = GodotRunner(project_dir='CATSINO.CASINO/godot')\npipeline_start = time.time()

In [ ]:
# Stage 1: Validate\nprint('=== STAGE 1: VALIDATE ===')\nv = runner.validate()\nif not v.success:\n    print('VALIDATION FAILED - aborting pipeline.')\n    for e in v.errors:\n        print(f'  {e}')\n    raise SystemExit(1)\nprint(f'Validation passed ({v.duration_seconds:.0f}s).')

In [ ]:
# Stage 2: Smoke\nprint('\\n=== STAGE 2: SMOKE ===')\ns = runner.boot_smoke()\nif not s.success:\n    print('SMOKE FAILED - aborting.')\n    raise SystemExit(1)\nprint('Smoke passed.')

In [ ]:
# Stage 3: Tests (optional)\nprint('\\n=== STAGE 3: TESTS ===')\nif os.path.exists('CATSINO.CASINO/godot/test'):\n    t = runner.run_tests()\n    print(f'Tests: {\"PASS\" if t.success else \"FAIL\"}')\nelse:\n    print('No tests - skipping.')

In [ ]:
# Stage 4: Export All\nprint('\\n=== STAGE 4: EXPORT ===')\nresults = runner.export_all()\nfailed = [n for n, r in results.items() if not r.success]\nif failed:\n    print(f'WARNING: {len(failed)} exports failed: {failed}')\nfor name, r in results.items():\n    print(f'  {name}: {\"PASS\" if r.success else \"FAIL\"} ({r.duration_seconds:.0f}s, {len(r.artifacts)} files)')

In [ ]:
# Stage 5: Package\nprint('\\n=== STAGE 5: PACKAGE ===')\nzip_path = runner.package_release(version=VERSION)\nif zip_path:\n    import os\n    size_mb = os.path.getsize(zip_path) / (1024*1024)\n    print(f'Release package: {zip_path} ({size_mb:.1f} MB)')\n    from google.colab import files\n    files.download(zip_path)\nelse:\n    print('Package failed.')

In [ ]:
total = time.time() - pipeline_start\nprint(f'\\n=== PIPELINE COMPLETE in {total:.0f}s ===')\nprint(f'Version: {VERSION}')